# MoodNote-AI — sensitivity: combined selected on VSMEC val (phase 4, post-hoc)

Runs on a **Colab T4 GPU**. Re-trains `combined` (same train data and hyperparameters as the ablation,
seeds 42/43/44) but picks the best epoch and early-stops on **VSMEC validation only**, the criterion
`real_only` uses; the ablation selected `combined` on VSMEC + synthetic validation pooled. Every epoch also
scores synthetic and pooled validation, so the report shows which epoch the pooled criterion would pick.

**Post-hoc, report-only**: decided after the ablation results were seen. The ablation verdict
(`comparison.json`) stays the main result; `sensitivity.md` reports this scenario next to it.

Each run is also tested with its best checkpoint loaded correctly (`*_fixed` scores): transformers 5.3
restores the best checkpoint without PhoBERT's LayerNorm (legacy gamma/beta names), and every ablation run
was tested that way. See `LOAD_NOTE` in `src/training/sensitivity_runner.py`.

Before you start:
- Runtime → Change runtime type → **T4 GPU**.
- Commit + push `src/training/sensitivity_runner.py` and this notebook
  (`git add -f notebooks/sensitivity_colab.ipynb`: every `*.ipynb` is gitignored by default).
- The ablation run JSONs must be on Drive in `MyDrive/MoodNote-AI/ablation/reports/`
  (`real_only_seed{42,43,44}.json`, `combined_seed{42,43,44}.json`): the runner stops before training if
  one is missing or was run with another config.
- Optional Colab secret `WANDB_API_KEY`; without it W&B logs offline.

Results go to Drive: `MyDrive/MoodNote-AI/ablation/sensitivity/` (one JSON per seed +
`sensitivity.{json,md}`), the seed-42 model to `MyDrive/MoodNote-AI/ablation/models/combined_vsmec_val/`.
About 45-50 min for the 3 runs. If the runtime disconnects, re-run Cell 1 and then the training cell:
finished seeds are skipped.

While training, transformers warns *"early stopping required metric_for_best_model, but did not find
eval_vsmec_f1_macro so early stopping is disabled"* twice per epoch. Expected: the callback runs once per
validation set and only counts the VSMEC one.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import os
import shutil
import subprocess

import torch
from google.colab import drive, userdata

BRANCH = "feature/ToanHuynh/NCKH-rebuild"
REPO_DIR = "/content/MoodNote-AI"
DRIVE_DIR = "/content/drive/MyDrive/MoodNote-AI/ablation"

if not torch.cuda.is_available():
    raise RuntimeError("GPU not found. Runtime -> Change runtime type -> T4 GPU")
print(f"GPU: {torch.cuda.get_device_name(0)}")

drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(
        ["git", "clone", "-b", BRANCH, "https://github.com/MoodNote/MoodNote-AI.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)

# Sensitivity results and the seed-42 model go to Drive. Results already committed to git are copied
# there first. The ablation run JSONs are read straight from Drive (--baseline-dir in Cell 4).
for local, remote in (
    ("reports/ablation_sensitivity", f"{DRIVE_DIR}/sensitivity"),
    ("models/ablation", f"{DRIVE_DIR}/models"),
):
    os.makedirs(remote, exist_ok=True)
    if not os.path.islink(local):
        if os.path.isdir(local):
            shutil.copytree(local, remote, dirs_exist_ok=True)
            shutil.rmtree(local)
        os.makedirs(os.path.dirname(local), exist_ok=True)
        os.symlink(remote, local)

In [ ]:
# ── Cell 2: Dependencies + W&B ───────────────────────────────────────────────
# Versions match requirements.txt; pyvi segments the VSMEC text like the local pipeline.
!pip install -q transformers==5.3.0 accelerate==1.13.0 scikit-learn==1.8.0 pyvi==0.1.1 wandb==0.25.0

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    os.environ["WANDB_MODE"] = "offline"
    print("No WANDB_API_KEY secret: W&B logs offline")

In [ ]:
# ── Cell 3: Data (UIT-VSMEC download + segmentation, ablation train/val files) ─
!python -m src.data.real.download_vsmec
!python -m src.data.real.preprocess
!python -m src.data.ablation

## Sensitivity runs

One JSON per seed in `reports/ablation_sensitivity/combined_vsmec_val_seed<S>.json`. To split the work
across sessions, run one seed at a time by adding e.g. `--seed 43`.

In [ ]:
# ── Cell 4: Train + evaluate (re-run after a disconnect: finished seeds are skipped) ─
!python -m src.training.sensitivity_runner --baseline-dir {DRIVE_DIR}/reports

In [ ]:
# ── Cell 5: Sensitivity report ───────────────────────────────────────────────
from IPython.display import Markdown, display

display(Markdown(open("reports/ablation_sensitivity/sensitivity.md", encoding="utf-8").read()))

## Next steps (local machine)

1. Download `MyDrive/MoodNote-AI/ablation/sensitivity/` into `reports/ablation_sensitivity/` and commit the
   JSON files (`sensitivity.md` is gitignored like every `*.md`; with all 3 run JSONs present,
   `python -m src.training.sensitivity_runner --baseline-dir <ablation run JSONs>` only re-summarizes).
2. The seed-42 model stays on Drive (`MyDrive/MoodNote-AI/ablation/models/combined_vsmec_val/`).